# 🚀 Module 53 — Model Serving with FastAPI & Streamlit

## 2. FastAPI

FastAPI is a Python framework used to build **REST APIs**. In ML deployment, we use it to expose our trained models so that external applications can send data and receive predictions.

## 1. REST Endpoints

An **endpoint** is a URL through which a client communicates with our backend.

| HTTP Method | Endpoint | Purpose |
|---|---|---|
| GET | `/` | Root endpoint |
| GET | `/health` | Check API status |
| POST | `/predict` | Send input for prediction |

```text
Client
  │
  │ HTTP Request
  ▼
FastAPI Endpoint
  │
  ▼
Validation
  │
  ▼
ML Model
  │
  ▼
Prediction
  │
  ▼
HTTP Response (JSON)
```

- **GET** → Retrieve information.
- **POST** → Send data for processing.
- **PUT** → Replace a resource.
- **PATCH** → Partially update a resource.
- **DELETE** → Delete a resource.

For ML inference, `POST` is generally used because the client sends input data in the request body.

## 2. Request Schemas

A request schema defines **what data the API expects from the client**.

FastAPI uses Pydantic models to define and validate incoming data.

```python
from pydantic import BaseModel

class PredictionRequest(BaseModel):
    text: str
```

Expected JSON input:

```json
{
    "text": "This movie was amazing!"
}
```

The schema specifies that:

- `text` is a required field.
- `text` must be a string.
- Incorrect input types are rejected automatically.

## 3. Response Schemas

A response schema defines **the structure of the output returned by the API**.

```python
from pydantic import BaseModel

class PredictionResponse(BaseModel):
    prediction: str
    confidence: float
```

Example response:

```json
{
    "prediction": "positive",
    "confidence": 0.98
}
```

### Why response schemas?

- Consistent output format
- Type validation
- Clear API contracts
- Automatic API documentation
- Easier frontend integration

## 4. Input Validation

Input validation ensures the incoming data is valid **before it reaches the model**.

Pydantic supports:

- Data type validation
- Required fields
- Minimum and maximum length
- Numeric ranges
- Custom validation rules

```python
from pydantic import BaseModel, Field

class PredictionRequest(BaseModel):
    text: str = Field(
        min_length=1,
        max_length=5000
    )
```

| Input | Result |
|---|---|
| `"This is amazing"` | ✅ Valid |
| `""` | ❌ Invalid |
| `12345` | ❌ Invalid type |
| Text exceeding 5000 characters | ❌ Invalid |

FastAPI automatically returns a validation error response when the request doesn't satisfy the schema.

## 5. Error Handling

APIs must handle errors safely instead of crashing unexpectedly.

| HTTP Status | Meaning |
|---|---|
| `200` | Successful request |
| `400` | Bad request |
| `422` | Input validation failure |
| `500` | Internal server error |
| `503` | Service unavailable |

### Custom error handling with `HTTPException`

```python
from fastapi import HTTPException

if not text.strip():
    raise HTTPException(
        status_code=400,
        detail="Text cannot be empty"
    )
```

Response:

```json
{
    "detail": "Text cannot be empty"
}
```

Never expose stack traces, sensitive data, credentials, or internal implementation details in production error responses.

## 6. Model Loading at Startup

The model should be loaded **once when the API starts**, rather than every time a prediction request arrives.

### ❌ Loading on every request

```python
@app.post("/predict")
def predict(request):
    model = load_model()
    return model.predict(request.text)
```

This causes unnecessary loading overhead, latency and memory-related inefficiency.

### ✅ Load once and reuse

```text
API Startup
     ↓
Load Model
     ↓
Keep Model in Memory
     ↓
Receive Requests
     ↓
Reuse Model
     ↓
Return Predictions
```

FastAPI's **lifespan** mechanism allows us to initialize resources during startup and handle cleanup during shutdown.

```python
from contextlib import asynccontextmanager
from fastapi import FastAPI

@asynccontextmanager
async def lifespan(app: FastAPI):
    app.state.model = load_model()
    yield
    # Cleanup during shutdown if necessary

app = FastAPI(lifespan=lifespan)
```

- `app.state.model` stores the model for use by endpoint functions.
- `yield` separates startup logic from shutdown logic.
- The model is reused across incoming requests rather than reloaded each time.

## 🧠 Quick Revision

| Concept | Purpose |
|---|---|
| REST endpoint | Exposes functionality through a URL |
| Request schema | Defines expected input |
| Response schema | Defines output structure |
| Pydantic | Validation and type enforcement |
| `HTTPException` | Controlled error responses |
| Lifespan | Startup and shutdown resource management |
| `app.state` | Store application-level resources |

**Core flow:**

```text
Client
  ↓
HTTP Request
  ↓
FastAPI Endpoint
  ↓
Pydantic Validation
  ↓
Preprocessing
  ↓
Model Inference
  ↓
Postprocessing
  ↓
Response Schema
  ↓
JSON Response
```

---
